In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import textwrap
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
#RF filter code using FFTs
def RF_filter(signal_array, filter_kernel, caption_text1, caption_text2, sampling_rate=1.0, show_graph=True):
    normalized_kernel = filter_kernel / np.sum(filter_kernel)
    filtered_signal = signal.fftconvolve(signal_array, normalized_kernel, mode='same')
    if show_graph:
        N = len(signal_array)
        t = np.arange(N) / sampling_rate
        M = len(filter_kernel)
        t_kernel = np.arange(M) / sampling_rate
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))
        plt.subplots_adjust(bottom=0.1, hspace=0.6)
        ax1.set_title("Fig 1. Input 1: Signal & Filter Kernel", fontweight='bold')
        ax1.plot(t, signal_array, color='steelblue', alpha=0.6, label='Input Signal (Left Axis)')
        ax1.tick_params(axis='y', labelcolor='steelblue')
        ax1_twin = ax1.twinx()
        ax1_twin.plot(t_kernel, normalized_kernel, color='#D62728', linewidth=2, label='Filter Kernal (Right Axis)')
        ax1_twin.fill_between(t_kernel, 0, normalized_kernel, color='#D62728', alpha=0.3)
        ax1_twin.tick_params(axis='y', labelcolor='#D62728')
        ax1_twin.set_ylim(bottom=0)
        ax1_twin.set_ylabel("Normalized Amplitude", color='#D62728')
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax1_twin.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')
        ax1.set_xlabel(f"Time (s)if Fs={sampling_rate} Hz")
        ax1.set_ylabel("Amplitude", color = 'steelblue')
        desc_1 = caption_text1 
        desc_2 = caption_text2
        ax1.text(0.5, -0.25, textwrap.fill(desc_1, width = 100), transform=ax1.transAxes, ha='center', va='top', fontsize=10, color='darkred', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.5))
        ax2.set_title("Fig 2. Output: Convolved Result", fontweight='bold')
        ax2.plot(t, signal_array, color='gray', alpha=0.3, label='(Original Input)')
        ax2.plot(t, filtered_signal, color='green', linewidth=2, label='Filtered Output')
        ax2.set_xlabel(f"Time (seconds) if Fs={sampling_rate}Hz")
        ax2.set_ylabel("Amplitude")
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)
        ax2.text(0.5, -0.3, textwrap.fill(desc_2, width = 100), transform=ax2.transAxes, ha='center', va='top', fontsize=10, color='darkgreen', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.5))
       
        plt.show()
    return filtered_signal


In [ ]:
#example for RF_filter function
from scipy.signal.windows import gaussian
fs = 1000 #1KHz
t = np.arange(2000) / fs

data = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 50 * t)

my_filter_shape = gaussian(100, std=15)
caption1 = "Fig. 1: The selected filter function shown in red and utilizing the right axis is convoluted with the original sample data shown in blue using the left axis. The filter function is normalized to ensure no amplitude shifts during convolution. The use of a gaussian filter in this exmaple smooths out the noise in the sample data, however you could select a number of filters such as a delta or sinc function."
caption2 = "Fig. 2: The resulting filtered and smoothed sample data in green relative to the original data in grey. The smoothed data does not have as high of an absolute amplitude as the orignal data due to the removal of noise. The gaussian kernel is a weighted moving average of a local area and is plotting that average as the filtered value. This works because noise is typically high frequency and random such that the sum of noise for local areas tends to be close to 0 which results in the main sin wave being revealed post filter."
clean_data = RF_filter(data, my_filter_shape, caption1, caption2, sampling_rate=fs)

print(f"Filtering complete. Output array shape: {clean_data.shape}") #do we get data back? check one check two

In [ ]:
def force_alias(signal_array, input_fs, target_fs, show_graph=True):
    M = int(input_fs / target_fs)
    aliased_signal = signal_array[::M]
    t_new = np.arange(len(aliased_signal)) / target_fs
    t_original = np.arange(len(signal_array)) / input_fs
   # print(len(aliased_signal), target_fs)
    #print(len(signal_array), input_fs)
    #print(t_new)
    #print(t_original)
    if show_graph:
        fig, ax = plt.subplots(figsize=(12, 6))
        plt.subplots_adjust(bottom=0.2) 
        ax.plot(t_original, signal_array, color='lightgray', linewidth=2, label=f'True Analog Signal ({input_fs} Hz)')
        ax.plot(t_new, aliased_signal, 'ro', label='Samples Taken')
        ax.plot(t_new, aliased_signal, 'r--', alpha=0.6, linewidth=2, label=f'Digital Interpretation ({target_fs} Hz)')
        ax.set_title(f"Visualizing Aliasing: {input_fs}Hz $\\rightarrow$ {target_fs}Hz", fontweight='bold')
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Amplitude")
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        desc = (
            f"Fig. 3: The red dots show the only data the computer sees. "
            f"Notice how they connect to form a wave that might be completely different "
            f"from the gray 'real' wave. This new wave is the alias."
        )
        wrapped_desc = textwrap.fill(desc, width=100)
        plt.figtext(0.5, 0.05, wrapped_desc, ha='center', fontsize=10, 
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.5))
        
        plt.show()
    
    return aliased_signal, t_new


In [ ]:
def id_signal_candidates(fig_num, observed, current_fs, filter_range=None, num_candidates=5, show_graph=True):
    #identifies potential true frequencies and highlights the one within the analog filter range
    #filter_range: tuple (f_min, f_max)
    candidates =[]
    for N in range(num_candidates+1):
        f_possible_1 = N*current_fs + observed
        f_possible_2 = N*current_fs - observed
        if f_possible_1 > 0: candidates.append(f_possible_1)
        if f_possible_2 > 0: candidates.append(f_possible_2)  
        #plt.figure(figsize=(12, 6))
        unique_candidates = sorted(list(set(candidates)))

    #logic for finding frequency within filter range
    match = None
    nyquist_zone = None
    if filter_range:
        f_min, f_max = filter_range
        for f in unique_candidates:
            if f_min <= f <= f_max:
                match = f
                nyquist_zone = int(np.ceil(f/(current_fs/2)))
                break

    if show_graph:
        fig, ax = plt.subplots(figsize=(12, 5))
        plt.subplots_adjust(bottom=0.3)

        #plots all candidates
        markerline, stemlines, baseline = ax.stem(unique_candidates, np.ones(len(unique_candidates)))
        plt.setp(markerline, color='red', marker='D', markersize=6, alpha=0.5)
        plt.setp(stemlines, color='red', linestyle='--', alpha = 0.3)

        #highlight filter range
        if filter_range:
            ax.axvspan(filter_range[0], filter_range[1], color='orange', alpha=0.15, label='Filter Passband') 
        #highlight frequency
        if match:
            ax.stem([match], [1], linefmt='r-', markerfmt='rD', basefmt=' ')
            ax.annotate(f'MATCH: {match} Hz\nZone {nyquist_zone}', xy=(match, 1), xytext=(match, 1.3), arrowprops=dict(facecolor='green', shrink=0.05), ha='center', fontweight='bold', color='green')
        if observed:
            ax.stem([observed], [1], linefmt='r-', markerfmt='rD', basefmt=' ')
            ax.annotate(f'OBSERVED: {observed} Hz', xy=(observed, 1), xytext=(observed, 1.3), arrowprops=dict(facecolor='blue', shrink=0.05), ha='center', fontweight='bold', color='blue')
        #ax.annotate('Observed\n(Alias)', xy=(observed, 1), xytext=(observed, 1.3),
                    #arrowprops=dict(facecolor='black', shrink=0.05), ha='center')
        ax.set_yticks([]) 
        ax.set_title(f"Fig {fig_num}. Signal Identification (Observed: {observed} Hz | $f_s$: {current_fs} Hz)", fontweight='bold')
        ax.set_xlabel("Frequency (Hz)", fontweight='bold')
        ax.set_ylim(0, 1.6)
        desc = (
        f"""Fig {fig_num}. You saw a signal at {observed} Hz with a sampling rate of {current_fs} Hz. If you used a bandpass filter then the highlighted frequency is the original frequency considering any aliasing observed and which Nyquist zone it resides in. If you did not use a bandpass filter, then this lists the possible frequencies associated with the sampled signal, which can be filtered manually using known physics of your source."""
        )
        wrapped_desc = textwrap.fill(desc, width=90)
        plt.figtext(0.5, 0.05, wrapped_desc, ha='center', fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.3", fc="#f0f0f0", ec="black", alpha=0.5))
        plt.legend(loc='upper right')
        print("-" * 30)
        print(f"ANALYSIS FOR {observed} Hz (Fs = {current_fs} Hz)")
        print(f"Potential Frequencies: {unique_candidates}")
        if match:
            print(f"IDENTIFIED SIGNAL: {match} Hz (Nyquist Zone {nyquist_zone})")
        print("-" * 30)
        plt.show()
    return {"all_candidates": unique_candidates, "identified_f": match, "nyquist_zone":nyquist_zone}

In [ ]:
#this is a test cell for the force_alias and _id_signal_candidates functions
result = id_signal_candidates(fig_num=1, observed=20, current_fs=100, filter_range=(210, 240))